# 03 - Is the model looking at the spot, or at something else?

Notebook 02 showed the naive split flatters the model. This notebook asks a second question:
**even on the fair split, how much of the score comes from the lesion itself?**

Three tests, all on the **fair split**:
1. **Hide the lesion.** Blank the middle of every photo and train on what is left (skin, hair, edges, rulers, borders). If it still scores well above chance, the model can use the background.
2. **Lesion only.** Blank everything *except* the middle. How much does the lesion alone give?
3. **Metadata only.** No image at all, just age, sex and body location. How far does that get?

Then **Grad-CAM heatmaps** show where the normal model actually looks.

**Not a medical tool.** Research audit of how models get fooled.

## Step 0 - Settings
- `SMOKE_TEST = True` runs a tiny version (a few minutes) just to check the code. Set to `False` for the real run.
- A real run trains **three** models. At about 14 minutes each, expect **about 45 minutes** with one seed.
- `SEEDS = [0]` first. Later `[0, 1, 2]` (about 2 hours) so you can report a mean and spread.
- `RADIUS = 0.40` is the size of the middle circle as a fraction of the photo width (so the circle covers 80% of the width).

In [ ]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import models
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score
from sklearn.linear_model import LogisticRegression

DATA = Path("data/archive-5")
SMOKE_TEST = True
SEEDS = [0]
EPOCHS = 5
BATCH = 64
LR = 3e-4
RADIUS = 0.40
VARIANTS = ["full", "no_lesion", "lesion_only"]

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("training on:", device)

## Step 1 - Load the cached photos and the fair split
Both come from notebooks 01 and 02, so run those first if `cache/` or `splits/` is missing.
We use **only the fair split** in this notebook.

In [ ]:
images = np.load("cache/images224.npy", mmap_mode="r")
row_of = {iid: i for i, iid in enumerate(json.load(open("cache/ids.json")))}

CLASSES = sorted(pd.read_csv(DATA / "HAM10000_metadata.csv").dx.unique())
label_of = {c: i for i, c in enumerate(CLASSES)}
CHANCE = 1 / len(CLASSES)

fair_train = pd.read_csv("splits/fair_train.csv"); fair_test = pd.read_csv("splits/fair_test.csv")
if SMOKE_TEST:
    fair_train, fair_test = fair_train.sample(400, random_state=0), fair_test.sample(200, random_state=0)
print(len(fair_train), "train /", len(fair_test), "test | chance balanced accuracy =", round(CHANCE, 3))

## Step 2 - The mask
A circle in the middle of the photo. Dermoscopy photos are taken with the lesion roughly centred,
so this is a simple stand-in for "the lesion".

**Honest limit:** HAM10000 does not come with outlines of each lesion, so this is an *approximation*.
Small lesions will leave extra skin inside the circle, and very large ones will poke outside it.
Say this in your write-up. Look at the picture below to judge it yourself.

In [ ]:
yy, xx = np.mgrid[:224, :224]
CIRCLE_NP = (((yy - 111.5) ** 2 + (xx - 111.5) ** 2) <= (RADIUS * 224) ** 2).astype(np.float32)
CIRCLE = torch.tensor(CIRCLE_NP, device=device).view(1, 1, 224, 224)
print(f"circle covers {CIRCLE_NP.mean():.0%} of the photo")

sample = [images[row_of[i]] for i in fair_test.image_id.iloc[:3]]
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for r, img in enumerate(sample):
    for c, (title, m) in enumerate([("full", np.ones_like(CIRCLE_NP)), ("no_lesion", 1 - CIRCLE_NP), ("lesion_only", CIRCLE_NP)]):
        axes[r, c].imshow((img * m[..., None] + 128 * (1 - m[..., None])).astype(np.uint8)); axes[r, c].axis("off")
        if r == 0: axes[r, c].set_title(title)
plt.tight_layout(); plt.show()

## Step 3 - Test 3 first: metadata only (no image, takes seconds)
Age, sex and body location. If this alone beats chance by a lot, those facts carry label information
that has nothing to do with how the lesion looks.

`dx_type` (how the diagnosis was confirmed) is left out on purpose. It is a by-product of the
diagnosis, not something known before it.

In [ ]:
def meta_features(tr, te):
    both = pd.concat([tr.assign(_s="tr"), te.assign(_s="te")])
    both["age"] = both.age.fillna(tr.age.median())
    X = pd.get_dummies(both[["age", "sex", "localization"]], columns=["sex", "localization"]).astype(float)
    X["age"] = (X.age - X.age.mean()) / X.age.std()
    return X[(both._s == "tr").values], X[(both._s == "te").values]

Xtr, Xte = meta_features(fair_train, fair_test)
y_tr = fair_train.dx.map(label_of).values; y_te = fair_test.dx.map(label_of).values
lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, y_tr)
p = lr.predict(Xte)
meta_res = {"variant": "metadata_only", "seed": 0, "accuracy": accuracy_score(y_te, p),
            "balanced_accuracy": balanced_accuracy_score(y_te, p), "macro_f1": f1_score(y_te, p, average="macro")}
print({k: round(v, 3) if isinstance(v, float) else v for k, v in meta_res.items()})

## Step 4 - The same model as notebook 02, with a mask option
Nothing changes from notebook 02 except one thing: `mode` decides which part of the photo the model is allowed to see.
Masked pixels are set to the average colour, so the blank area carries no information.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def batch_tensor(rows, augment, mode):
    x = torch.from_numpy(np.stack([images[row_of[i]] for i in rows])).to(device)
    x = x.permute(0, 3, 1, 2).float() / 255
    if augment:
        h = torch.rand(len(x), device=device) < 0.5
        v = torch.rand(len(x), device=device) < 0.5
        x = torch.where(h.view(-1, 1, 1, 1), x.flip(3), x)
        x = torch.where(v.view(-1, 1, 1, 1), x.flip(2), x)
    x = (x - MEAN) / STD
    if mode == "no_lesion":   x = x * (1 - CIRCLE)
    if mode == "lesion_only": x = x * CIRCLE
    return x

def make_model():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, len(CLASSES))
    return m.to(device)

def predict(model, df, mode):
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(df), 128):
            out.append(model(batch_tensor(df.image_id.iloc[s:s+128], False, mode)).argmax(1).cpu())
    return torch.cat(out).numpy()

def run(mode, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    tr, te = fair_train, fair_test
    y_tr = tr.dx.map(label_of).values; y_te = te.dx.map(label_of).values
    counts = np.bincount(y_tr, minlength=len(CLASSES)).clip(min=1)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(len(y_tr) / (len(CLASSES) * counts), dtype=torch.float, device=device))
    model = make_model(); opt = torch.optim.AdamW(model.parameters(), lr=LR)
    epochs = 1 if SMOKE_TEST else EPOCHS
    for ep in range(epochs):
        model.train(); order = np.random.permutation(len(tr)); total = 0; t0 = time.time()
        for s in range(0, len(order), BATCH):
            idx = order[s:s+BATCH]
            x = batch_tensor(tr.image_id.iloc[idx], True, mode); y = torch.tensor(y_tr[idx], device=device)
            opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        print(f"  [{mode} seed {seed}] epoch {ep+1}/{epochs}  loss {total/len(tr):.3f}  ({time.time()-t0:.0f}s)")
    pred = predict(model, te, mode)     # test set used once, at the end
    res = {"variant": mode, "seed": seed, "accuracy": accuracy_score(y_te, pred),
           "balanced_accuracy": balanced_accuracy_score(y_te, pred), "macro_f1": f1_score(y_te, pred, average="macro")}
    return res, model

## Step 5 - Run the three variants
This is the long cell (about 45 minutes with one seed). The "full" model is saved so Step 7 can draw heatmaps from it.

In [ ]:
Path("results").mkdir(exist_ok=True)
rows, full_model = [], None
for seed in SEEDS:
    for mode in VARIANTS:
        res, model = run(mode, seed)
        rows.append(res)
        if mode == "full" and full_model is None:
            full_model = model
            if not SMOKE_TEST:
                torch.save(model.state_dict(), "results/resnet18_full_fair.pt")
rows.append(meta_res)
res = pd.DataFrame(rows)
res.to_csv("results/smoke_03.csv" if SMOKE_TEST else "results/03_scores.csv", index=False)
res.round(3)

## Step 6 - Compare
How to read the chart:
- **Dotted line = chance** (about 0.14 for 7 classes). A model at that line learned nothing.
- **full** is the normal model on the fair split. It should land near the fair number from notebook 02, give or take run-to-run noise.
- **no_lesion** well above chance means the **background carries information about the diagnosis**. That is the shortcut finding.
- **no_lesion near chance** is also a result: the model cannot cheat from the background (with this mask).
- **lesion_only** versus **full** tells you how much the surroundings add on top of the lesion.

In [ ]:
order = ["full", "lesion_only", "no_lesion", "metadata_only"]
summ = res.groupby("variant").balanced_accuracy.agg(["mean", "std"]).reindex(order)
display(res.groupby("variant")[["accuracy", "balanced_accuracy", "macro_f1"]].mean().reindex(order).round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(order, summ["mean"], yerr=summ["std"].fillna(0), color=["#2e7d32", "#1565c0", "#d9534f", "#8e8e8e"])
ax.axhline(CHANCE, ls=":", color="k"); ax.text(3.45, CHANCE + 0.01, "chance", ha="right")
ax.set_ylim(0, 1); ax.set_ylabel("balanced accuracy (fair split)"); ax.set_title("What the model can see vs how well it does")
plt.tight_layout()
if not SMOKE_TEST: plt.savefig("results/03_shortcut_tests.png", dpi=150)
plt.show()

## Step 7 - Grad-CAM: where does the normal model look?
Grad-CAM colours the parts of a photo that pushed the model toward its answer (red = most influence).
Two things to look at:
- **The pictures:** does the red sit on the spot, or on the corners, edges, hair or rulers?
- **One number:** the share of that attention that falls inside the middle circle. If the model looked evenly everywhere, that share would equal the circle's share of the photo (about 50%). Much higher means it focuses on the middle. Much lower means it mostly looks at the surroundings.

Grad-CAM is a rough guide, not proof. A heatmap on the spot does not show the model *understands* the spot.

In [ ]:
def gradcam(model, x):
    model.eval(); feats = {}
    h = model.layer4.register_forward_hook(lambda m, i, o: feats.update(a=o))
    logits = model(x); feats["a"].retain_grad()
    cls = logits.argmax(1)
    logits.gather(1, cls[:, None]).sum().backward()
    a = feats["a"]; w = a.grad.mean((2, 3), keepdim=True)
    cam = F.interpolate(torch.relu((w * a).sum(1, keepdim=True)), size=224, mode="bilinear", align_corners=False)
    cam = cam / cam.amax((2, 3), keepdim=True).clamp(min=1e-8)
    h.remove(); model.zero_grad()
    return cam.detach().cpu()[:, 0].numpy(), cls.cpu().numpy()

if full_model is None:   # kernel was restarted: load the saved weights
    full_model = make_model(); full_model.load_state_dict(torch.load("results/resnet18_full_fair.pt", map_location=device))

sub = fair_test.sample(min(300, len(fair_test)), random_state=0)
shares = []
for s in range(0, len(sub), 50):
    cam, _ = gradcam(full_model, batch_tensor(sub.image_id.iloc[s:s+50], False, "full"))
    shares.append((cam * CIRCLE_NP).sum((1, 2)) / cam.sum((1, 2)).clip(min=1e-8))
shares = np.concatenate(shares)
print(f"attention inside the middle circle: {shares.mean():.0%} on average")
print(f"(the circle is {CIRCLE_NP.mean():.0%} of the photo, so evenly spread attention would give {CIRCLE_NP.mean():.0%})")

In [ ]:
picks = pd.concat([fair_test[fair_test.dx == d].head(2) for d in ["mel", "nv", "bkl", "bcc"]])
cam, pred = gradcam(full_model, batch_tensor(picks.image_id, False, "full"))
fig, axes = plt.subplots(2, 8, figsize=(20, 5.5))
for i, (_, r) in enumerate(picks.iterrows()):
    img = images[row_of[r.image_id]]
    axes[0, i].imshow(img); axes[0, i].axis("off")
    axes[0, i].set_title(f"true {r.dx}\npred {CLASSES[pred[i]]}", fontsize=9, color="green" if CLASSES[pred[i]] == r.dx else "red")
    axes[1, i].imshow(img); axes[1, i].imshow(cam[i], cmap="jet", alpha=0.45); axes[1, i].axis("off")
plt.tight_layout()
if not SMOKE_TEST: plt.savefig("results/03_gradcam.png", dpi=130)
plt.show()

### Write down for your README
- full / lesion_only / no_lesion / metadata_only balanced accuracy: ____ / ____ / ____ / ____ (chance 0.143)
- attention inside the middle circle: ____% (even spread would be ____%)
- what the heatmaps show (be literal, e.g. "mostly the lesion", "corners on 3 of 8"): ____

**How to word the result honestly**
- `no_lesion` clearly above chance: "a model that never sees the middle of the photo still reaches X, so the background carries signal." Add the mask caveat.
- `no_lesion` near chance: "with the middle hidden, the model falls to chance, so it was not cheating from the background in this dataset."
- Either way, say the mask is an approximation, it was one seed (or three), and this is not a medical tool.

**Next (notebook 04):** repeat with 3 seeds, then write the README and the LinkedIn post.